# Phase 2.6 — Cluster-then-Predict (Novelty Layer)

**Hypothesis:** Global tree models hit a ~0.76 AUC ceiling because customer behavior is bimodal — active customers and reactivators follow opposite recency-churn dynamics. Segment-specific models can break this ceiling by fitting each cluster's local structure without global compromises.

**Architecture:**
- Stage 1: K-means on behavioral features (train only) to discover segments
- Stage 2: Per-cluster XGBoost models using Phase 2.3 best params (no re-tuning)
- Comparison: aggregated cluster-predict AUC vs global XGBoost on the same frozen test set

**Rubric tie-in:** §5.4 Optimization & Novelty — the central novelty contribution. Direct comparison with prior related work on cluster-then-predict churn architectures. Per-cluster feature heterogeneity provides evidence of genuine segment-conditional structure.

**Expected runtime:** ~45 min (K selection: 5 fits; per-cluster training: K models). Short-circuits via `cluster_predict_results.json` on re-run.

**Last updated:** 2026-05-04

In [ ]:
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import Image, display

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, auc as sk_auc,
    silhouette_score,
)
import xgboost as xgb

warnings.filterwarnings('ignore')
os.environ['PYTHONIOENCODING'] = 'utf-8'

sys.path.insert(0, os.path.abspath('..'))
from src.config import (
    TRAIN_PATH, TEST_PATH, FIGURES, MODELS, TABLES,
    BENCHMARK_CSV, RANDOM_SEED
)
from src.evaluate import evaluate, print_metrics

FIG_DIR = FIGURES / '06_cluster_predict'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

## 1. Setup — Load Data & Preprocessors

Same frozen train/test as all prior phases. The XGBoost preprocessor from Phase 2.3 is shared across all cluster models — fit on the full training set, reused per cluster. This prevents encoding drift across cluster boundaries.

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

TARGET   = 'churned'
METADATA = ['wallet_id', 'full_name', 'churned_vendor']

ALL_FEATURE_COLS = [c for c in train.columns if c not in [TARGET] + METADATA]
CATEGORICAL_COLS = [
    'gender', 'state', 'city', 'referral_source',
    'preferred_language', 'linked_bank', 'kyc_tier',
]

CLUSTER_FEATURES = [
    'txn_count', 'total_amount_ngn', 'avg_amount_ngn',
    'days_since_last_txn', 'account_active_span_days',
    'txn_velocity_change', 'txn_per_active_day',
    'unique_channels', 'unique_txn_types',
    'is_reactivator', 'dormancy_days_before_reactivation',
    'tenure_days', 'fraud_rate',
]

y_train = train[TARGET].values
y_test  = test[TARGET].values

# Shared XGBoost preprocessor from Phase 2.3
global_pipe      = joblib.load(MODELS / 'xgboost.pkl')
xgb_preprocessor = global_pipe.named_steps['preprocessor']
global_clf       = global_pipe.named_steps['classifier']
feat_names       = xgb_preprocessor.get_feature_names_out()
X_train_enc = xgb_preprocessor.transform(train[ALL_FEATURE_COLS])
X_test_enc  = xgb_preprocessor.transform(test[ALL_FEATURE_COLS])

# Clustering StandardScaler — fit on train only
cluster_scaler  = StandardScaler()
X_train_cluster = cluster_scaler.fit_transform(train[CLUSTER_FEATURES].values)
X_test_cluster  = cluster_scaler.transform(test[CLUSTER_FEATURES].values)

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Clustering features: {len(CLUSTER_FEATURES)}')
print(f'XGBoost features: {len(feat_names)}')

## 2. Stage 1 — K-means Clustering

**Feature selection rationale:** Using 13 behavioral features rather than all 34. High-cardinality geographic features (state, city: 37 states) add noise in Euclidean distance space without contributing to behavioral segmentation. The 13 features capture the dimensions that matter: transaction volume, recency, activity velocity, reactivation status, and engagement diversity.

**StandardScaler:** Required for K-means — unscaled `total_amount_ngn` (range: 0–millions) would dominate the distance metric over `fraud_rate` (range: 0–1).

**K selection:** Elbow (inertia) + silhouette for K=2..6. Silhouette score guides the final choice; K=4 is the interpretability default when the margin is < 0.02 — that roughly corresponds to: heavy actives, light actives, reactivators, low-engagement.

In [ ]:
results_path = TABLES / 'cluster_predict_results.json'
if results_path.exists():
    with open(results_path, encoding='utf-8') as f:
        _res = json.load(f)
    CHOSEN_K   = _res['chosen_k']
    K_RANGE    = _res['k_range']
    inertias   = _res['inertias']
    sil_scores = _res['silhouette_scores']
    print(f'Loaded pre-computed results (K={CHOSEN_K})')
    print(f'Silhouette scores: {[round(s,4) for s in sil_scores]}')
else:
    # Run K selection from scratch (~5-10 min)
    K_RANGE, inertias, sil_scores = [], [], []
    rng = np.random.default_rng(RANDOM_SEED)
    sil_idx = rng.choice(len(X_train_cluster), size=30000, replace=False)
    X_sil   = X_train_cluster[sil_idx]
    for k in [2,3,4,5,6]:
        km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
        labels_full = km.fit_predict(X_train_cluster)
        inertias.append(km.inertia_)
        sil = silhouette_score(X_sil, labels_full[sil_idx])
        sil_scores.append(sil)
        K_RANGE.append(k)
        print(f'K={k}: silhouette={sil:.4f}')
    best_k_idx = int(np.argmax(sil_scores[1:]))
    best_k_c   = K_RANGE[1:][best_k_idx]
    CHOSEN_K   = 4 if abs(sil_scores[1:][best_k_idx] - sil_scores[K_RANGE.index(4)]) < 0.02 else best_k_c

In [ ]:
display(Image(str(FIG_DIR / 'kmeans_k_selection.png')))

In [ ]:
kmeans_path = MODELS / 'kmeans.pkl'
if kmeans_path.exists():
    kmeans = joblib.load(kmeans_path)
    print(f'Loaded saved kmeans (K={kmeans.n_clusters})')
else:
    kmeans = KMeans(n_clusters=CHOSEN_K, random_state=RANDOM_SEED, n_init=20)
    kmeans.fit(X_train_cluster)
    joblib.dump(kmeans, kmeans_path)
    print(f'Fitted K={CHOSEN_K} KMeans, saved to kmeans.pkl')

train_clusters = kmeans.predict(X_train_cluster)
test_clusters  = kmeans.predict(X_test_cluster)

print(f'Train cluster distribution: {np.bincount(train_clusters).tolist()}')
print(f'Test  cluster distribution: {np.bincount(test_clusters).tolist()}')

### Cluster Profiling

For each cluster: size, churn rate, and mean of the key behavioral features. The churn rate per cluster is the most important output — it tells us whether K-means found groups with genuinely different churn dynamics.

In [ ]:
profiles_path = TABLES / 'cluster_profiles.csv'
if profiles_path.exists():
    profiles_df = pd.read_csv(profiles_path)
    print('Loaded cluster_profiles.csv')
else:
    profile_rows = []
    for k in range(CHOSEN_K):
        mask = train_clusters == k
        df_k = train[mask]
        row  = {'cluster': k, 'n_train': int(mask.sum()),
                'pct_train': round(mask.mean()*100, 2),
                'n_churners_train': int(y_train[mask].sum()),
                'churn_rate': round(float(y_train[mask].mean()), 4)}
        for feat in CLUSTER_FEATURES:
            row[f'mean_{feat}'] = round(float(df_k[feat].mean()), 4)
        profile_rows.append(row)
    profiles_df = pd.DataFrame(profile_rows)
    profiles_df.to_csv(profiles_path, index=False)

disp_cols = ['cluster','n_train','pct_train','n_churners_train','churn_rate',
             'mean_txn_count','mean_days_since_last_txn',
             'mean_is_reactivator','mean_dormancy_days_before_reactivation',
             'mean_txn_velocity_change']
display(profiles_df[disp_cols].round(3))

### Cluster Labels

Based on the actual profiles from `run_cluster_predict.py`:

| Cluster | Name | N (train) | Churn rate | Key signal |
|---|---|---|---|---|
| **0** | Regular Active Customers | 293,532 (97.7%) | 8.7% | 26.7 mean txns, 44.6 days recency, zero reactivators |
| **1** | Dormant Reactivators | 4,589 (1.5%) | **100%** | is_reactivator=1.0, 194.5 days mean dormancy — every customer churned |
| **2** | Very Light Users | 2,308 (0.8%) | 11.3% | 3.35 mean txns (vs 26.7 for Cluster 0), near-zero reactivators |

Cluster 1 is the most striking result. K-means found a 100%-churn segment from behavioral features alone — zero label information was used in the clustering. The dormancy and reactivation features are the defining signal: mean 194.5 dormancy days (more than 6 months inactive before a brief return) maps to every one of these customers churning in the label window.

Cluster 2 is the low-activity outlier: very few transactions, no dormancy structure, moderate churn. These are customers who joined and barely engaged — not former active users who went dormant.

## 3. Stage 2 — Per-Cluster XGBoost Models

**Design choice:** Reuse Phase 2.3 Optuna best params exactly — no re-tuning. This isolates the segmentation effect from the tuning effect. Any AUC delta vs the global model is attributable to segmentation alone, not to better hyperparameters.

**Fixed n_estimators:** Set to the global model's `best_iteration` (249) rather than running early stopping per cluster. Cluster training sets are smaller (~25–100K rows), and early stopping on a 10% val split of a small cluster would be too noisy to trust. 249 trees at lr=0.01 is the global model's settled optimum — it's the right starting point for each segment.

**scale_pos_weight:** Recomputed per cluster from its own class ratio. Clusters with higher churn rates get lower scale_pos_weight — the model calibrates to its own base rate.

**Fallback rule:** If a cluster has < 1,000 training customers OR < 100 churners, the global model is used for that cluster's predictions. Documented explicitly.

In [ ]:
with open(TABLES / 'xgb_results.json', encoding='utf-8') as f:
    xgb_results  = json.load(f)
best_params  = xgb_results['best_params']
best_iter    = xgb_results['best_iter_tuned']
print(f'Phase 2.3 best params loaded: n_est_fixed={best_iter}, '
      f'depth={best_params["max_depth"]}, lr={best_params["learning_rate"]:.5f}')

MIN_SIZE = 1000; MIN_CHURN = 100
cluster_models   = {}
cluster_fallback = {}
feat_idx_map = {f'f{i}': name for i, name in enumerate(feat_names)}

for k in range(CHOSEN_K):
    pkl_path = MODELS / f'xgb_cluster_{k}.pkl'
    mask_tr  = train_clusters == k
    n_tr     = int(mask_tr.sum())
    n_churn  = int(y_train[mask_tr].sum())
    print(f'\nCluster {k}: n_train={n_tr}, churners={n_churn}')

    # Pure-class: XGBoost cannot train binary classifier on a single class
    is_pure = (n_churn == n_tr or n_churn == 0)
    if n_tr < MIN_SIZE or n_churn < MIN_CHURN or is_pure:
        reason = ("pure-class (all churned)" if n_churn == n_tr
                  else "pure-class (no churners)" if n_churn == 0
                  else "below size/churn threshold")
        print(f'  -> FALLBACK to global model ({reason})')
        cluster_models[k] = None; cluster_fallback[k] = True
        continue

    cluster_fallback[k] = False
    if pkl_path.exists():
        pipe_k = joblib.load(pkl_path)
        cluster_models[k] = pipe_k.named_steps['classifier']
        print(f'  Loaded from {pkl_path.name}')
        continue
    X_k = X_train_enc[mask_tr]; y_k = y_train[mask_tr]
    spw_k = int((y_k==0).sum()) / int((y_k==1).sum())
    params_k = {**best_params, 'n_estimators': best_iter,
                'scale_pos_weight': spw_k, 'tree_method': 'hist',
                'eval_metric': 'auc', 'n_jobs': -1,
                'random_state': RANDOM_SEED, 'verbosity': 0}
    clf_k = xgb.XGBClassifier(**params_k)
    t0 = time.time()
    clf_k.fit(X_k, y_k, verbose=False)
    cluster_models[k] = clf_k
    pipe_k = Pipeline([('preprocessor', xgb_preprocessor), ('classifier', clf_k)])
    joblib.dump(pipe_k, pkl_path)
    print(f'  Trained in {time.time()-t0:.1f}s | saved {pkl_path.name}')

## 4. Evaluation — Core Comparison

Three evaluation views:
1. **Full test AUC** — aggregated cluster-routed predictions vs global XGBoost
2. **Per-cluster AUC** — each cluster's model evaluated on its own test slice
3. **Per-cluster lift** — cluster-specific model vs global XGBoost *restricted to the same customers*

View 3 is the critical one. It shows whether segment-specific training helped *within* each segment, not just whether one segment happens to be easy.

In [ ]:
# Global model predictions
y_prob_global = global_clf.predict_proba(X_test_enc)[:, 1]
auc_global    = roc_auc_score(y_test, y_prob_global)
print(f'Global XGB AUC: {auc_global:.4f}')

# Cluster-routed predictions
y_prob_cluster = np.zeros(len(y_test))
for k in range(CHOSEN_K):
    mask_te = test_clusters == k
    if not mask_te.any(): continue
    if cluster_fallback[k]:
        y_prob_cluster[mask_te] = global_clf.predict_proba(X_test_enc[mask_te])[:, 1]
    else:
        y_prob_cluster[mask_te] = cluster_models[k].predict_proba(X_test_enc[mask_te])[:, 1]

res_cluster = evaluate(y_test, y_prob_cluster)
print_metrics(res_cluster, 'ClusterPredict_XGB')

print(f'\nDelta vs Global XGB:  {res_cluster["auc"] - auc_global:+.4f}')
print(f'Ceiling broken (>0.760): {res_cluster["auc"] > 0.760}')

In [ ]:
pc_path = TABLES / 'per_cluster_results.csv'
if pc_path.exists():
    per_cluster_df = pd.read_csv(pc_path)
    print('Loaded per_cluster_results.csv')
else:
    rows = []
    for k in range(CHOSEN_K):
        mask_te = test_clusters == k
        y_te_k  = y_test[mask_te]
        if not mask_te.any() or y_te_k.sum() == 0:
            rows.append({'cluster':k,'n_test':int(mask_te.sum()),
                         'auc_cluster_model':None,'auc_global_restricted':None,
                         'lift':None,'fallback':cluster_fallback[k]})
            continue
        proba_cm = global_clf.predict_proba(X_test_enc[mask_te])[:,1] if cluster_fallback[k] \
                   else cluster_models[k].predict_proba(X_test_enc[mask_te])[:,1]
        proba_gl = global_clf.predict_proba(X_test_enc[mask_te])[:,1]
        auc_cm   = roc_auc_score(y_te_k, proba_cm)
        auc_gl   = roc_auc_score(y_te_k, proba_gl)
        rows.append({'cluster':k,'n_test':int(mask_te.sum()),
                     'churn_rate':round(float(y_te_k.mean()),4),
                     'auc_cluster_model':round(auc_cm,4),
                     'auc_global_restricted':round(auc_gl,4),
                     'lift':round(auc_cm-auc_gl,4),'fallback':cluster_fallback[k]})
    per_cluster_df = pd.DataFrame(rows)
    per_cluster_df.to_csv(pc_path, index=False)

display(per_cluster_df)

In [ ]:
def safe_append(model_name, results, notes=''):
    row = {'model': model_name, **results, 'notes': notes}
    df_new = pd.DataFrame([row])
    if BENCHMARK_CSV.exists():
        df = pd.read_csv(BENCHMARK_CSV)
        if model_name in df['model'].values:
            print(f'[benchmark] {model_name} already present — skipping.')
            return
        df = pd.concat([df, df_new], ignore_index=True)
    else:
        df = df_new
    df.to_csv(BENCHMARK_CSV, index=False)
    print(f'[benchmark] appended {model_name}')

fallback_list   = [k for k in range(CHOSEN_K) if cluster_fallback[k]]
trained_list    = [k for k in range(CHOSEN_K) if not cluster_fallback[k]]
notes_c = (f'K={CHOSEN_K} KMeans; {len(trained_list)} cluster models '
           f'(fallback: {fallback_list}); XGB Phase 2.3 params, n_est=249 fixed')
safe_append('ClusterPredict_XGB', res_cluster, notes=notes_c)

## 5. Per-Cluster Lift Plot

For each cluster: cluster-specific model AUC vs global XGBoost restricted to the same test slice. A positive bar means segmentation helped for that cluster. The dashed line shows global XGB on the full test set for reference.

In [ ]:
display(Image(str(FIG_DIR / 'per_cluster_lift.png')))

## 6. ROC and PR Comparison

Three-model comparison: Global XGBoost, RF Tuned, ClusterPredict. The ROC curves should overlap closely if segmentation adds marginal global AUC. The PR curve is more informative at 10.1% churn — look for shifts at high-recall operating points.

In [ ]:
display(Image(str(FIG_DIR / 'roc_pr_comparison.png')))

## 7. Feature Importance Heatmap — Segment Heterogeneity

If different clusters depend on different features, that is direct evidence that K-means found segments with distinct churn dynamics — not just arbitrary partitions. The heatmap shows normalized gain (0–1 per model column) for the top 15 features from the global model.

In [ ]:
display(Image(str(FIG_DIR / 'cluster_feature_importance.png')))

### Feature Heterogeneity Interpretation

The heatmap has three model columns: Global XGB, Cluster 0 (regular actives, 293K train), and Cluster 2 (very light users, 2.3K train). Cluster 1 uses the global model by fallback.

Cluster 0 vs Global XGB: the columns look similar because Cluster 0 is 97.7% of the training data — the global model is essentially a Cluster 0 model with a small reactivator correction. The feature ranking shifts slightly: `days_since_last_txn` gains relative weight in the Cluster 0 model because `dormancy_days_before_reactivation` is absent (no reactivators in this cluster).

Cluster 2 vs Global XGB: the columns differ more visibly. `txn_count` and `account_active_span_days` are brighter for Cluster 2 — the model leans on total activity volume because that's what separates the very light users from each other. `dormancy_days_before_reactivation` and `is_reactivator` are near-zero (not present in this cluster). `tenure_days` and `txn_velocity_change` show elevated importance, suggesting that engagement trajectory matters more than recency for this low-use group.

This is moderate evidence of segment-conditional structure: the feature rankings aren't identical, but they aren't dramatically different either. The global model already accommodates these differences reasonably well through its non-linear splits.

## 8. K Selection — Supporting Analysis

Elbow (inertia) and silhouette scores for K=2..6.

In [ ]:
display(Image(str(FIG_DIR / 'kmeans_k_selection.png')))

## 9. DBSCAN Sensitivity Check (Optional)

K-means assumes spherical clusters of equal density. DBSCAN is density-based and discovers clusters of arbitrary shape without requiring K. If DBSCAN finds fundamentally different structure, it would undermine the K-means interpretation.

In [ ]:
from sklearn.cluster import DBSCAN

# Run on a 10% sample to keep it tractable (DBSCAN is O(n^2) on dense data)
rng = np.random.default_rng(RANDOM_SEED)
sample_idx = rng.choice(len(X_train_cluster), size=30000, replace=False)
X_db_sample = X_train_cluster[sample_idx]
y_db_sample = y_train[sample_idx]

t0 = time.time()
dbscan = DBSCAN(eps=0.5, min_samples=50, n_jobs=-1)
db_labels = dbscan.fit_predict(X_db_sample)
elapsed_db = time.time() - t0

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise       = int((db_labels == -1).sum())
pct_noise     = n_noise / len(db_labels) * 100

print(f'DBSCAN done in {elapsed_db:.1f}s')
print(f'DBSCAN found {n_clusters_db} clusters + {n_noise} noise points ({pct_noise:.1f}%)')
print(f'K-means chose K={CHOSEN_K}')

if n_clusters_db > 0:
    for k_db in range(n_clusters_db):
        mask_k = db_labels == k_db
        n_k    = int(mask_k.sum())
        cr_k   = float(y_db_sample[mask_k].mean())
        print(f'  DBSCAN cluster {k_db}: n={n_k} ({n_k/len(db_labels)*100:.1f}%)  churn={cr_k:.4f}')

## 10. Cluster-then-Predict — Findings

### Chosen K and Silhouette

K=3, auto-selected. Silhouette scores: K=2 → 0.685, K=3 → 0.670, K=4 → 0.113, K=5 → 0.113, K=6 → 0.110. The sharp drop from K=3 to K=4 (0.670 → 0.113) is the key signal — the data has 3 natural behavioral segments and K-means finds no further coherent structure beyond that. The high silhouette at K=2 and K=3 (0.67–0.69) confirms tight, well-separated clusters, not noise partitions.

---

### Cluster Profiles

**Cluster 0 — Regular Active Customers (97.7%, 8.7% churn):** The mainstream customer base. 26.7 mean transactions, 44.6 days mean recency. Zero reactivators. The vast majority of prediction decisions are made here.

**Cluster 1 — Dormant Reactivators (1.5%, 100% churn):** This cluster is the most significant unsupervised finding. K-means identified these 4,589 customers using only behavioral features — no label information. Mean dormancy before reactivation: 194.5 days (over 6 months). Every single one of them churned in the label window. This is near-perfect label purity from unsupervised clustering alone.

**Cluster 2 — Very Light Users (0.8%, 11.3% churn):** Customers with only 3.35 mean transactions (vs 26.7 for Cluster 0). Not dormant reactivators — their dormancy metrics are near-zero. They joined but barely used the service. Moderate churn at 11.3%.

---

### Aggregated AUC vs Global XGBoost

**ClusterPredict AUC: 0.7585 vs Global XGB: 0.7590 — delta: −0.0005.**

The 0.76 ceiling was not broken. Cluster-then-predict produced a result marginally below the global model. The AUC difference is within noise.

---

### Which Clusters Benefited, and Why

| Cluster | N (test) | Cluster model AUC | Global restricted AUC | Lift |
|---|---|---|---|---|
| 0 — Regular Actives | 73,447 | 0.7161 | 0.7161 | +0.0001 |
| 1 — Reactivators | 1,139 | NaN (fallback) | NaN (all churned) | — |
| 2 — Very Light | 522 | 0.7298 | 0.7636 | **−0.0338** |

Cluster 0 showed zero lift. Training on 97.7% of the data is functionally equivalent to training on 100% — the global model already fits this segment as well as a dedicated model can.

Cluster 2 showed *negative* lift (−0.034). The cluster-specific model underperformed the global model on the same 522 test customers. The cause is clear: 2,308 training samples is too small for 249 trees at lr=0.01, especially at 11.3% churn (260 churners). The global model, trained on 130× more data, generalises better even on this small population.

Cluster 1 is not evaluable (all 1,139 test customers churned — AUC is undefined when only one class is present). For production, this cluster doesn't need a model — it needs an automatic trigger.

---

### Feature Heterogeneity Finding

The heatmap shows mild heterogeneity. Cluster 0 looks nearly identical to the global model (expected — it's 97.7% of training). Cluster 2 shows higher relative importance for `txn_count` and `account_active_span_days` and near-zero `dormancy_days_before_reactivation` — the right features for a low-activity, non-reactivator group. This confirms that K-means found behaviorally distinct segments, but the global model's non-linear splits already approximate these distinctions without explicit segmentation.

---

### Hypothesis Verdict

**Not validated.** The 0.76 AUC ceiling was not broken.

The bimodal behavioral structure (active vs. reactivator) is real and was cleanly captured by K-means — Cluster 1's 100% churn rate from unsupervised clustering is a genuine finding. But this structure was *already being captured* by the global model through the `is_reactivator` and `dormancy_days_before_reactivation` features, which rank 1st and 2nd by gain in Phase 2.3. Explicit segmentation added no information that the global model's feature set didn't already encode.

What the hypothesis got right: the bimodal structure exists, and it drives the LR-to-tree gap (+0.051 AUC). What it missed: the global XGBoost already handles that bimodality through its tree splits — it doesn't need separate models to accommodate it.

---

### Implication for Production (Whish/OMT)

Three routing strategies replace the single-model approach:

1. **Cluster 1 — Dormant Reactivators:** No model needed. 100% historical churn rate — trigger a reactivation campaign automatically for any customer assigned to this cluster. The K-means routing rule is a deployable product feature: detect dormancy patterns → flag for reactivation intervention.

2. **Cluster 0 — Regular Active Customers (97.7%):** Use the global XGBoost model. The dedicated cluster model provides no lift.

3. **Cluster 2 — Very Light Users (0.8%):** Use the global XGBoost model. Too small for a reliable dedicated model. Separately, the business should evaluate whether retention spend on this segment is cost-effective given their very low historical transaction volume.

The architecture's value is not in AUC improvement — it's in operationalising different retention strategies per segment. A single model outputs a probability score; a cluster-routed system outputs both a score *and* a segment identity that informs which retention action to take. That's the differentiation for the Whish pitch.

## 11. Benchmark Summary

In [ ]:
bench = pd.read_csv(BENCHMARK_CSV)
display(bench[['model','auc','pr_auc','f1','precision','recall',
               'precision_at_k','recall_at_k']].round(4))